# 05 — Monthly Pairs Trading on JKP Stocks

This notebook builds a **memory-conscious, monthly pairs trading workflow** using the stock-level JKP dataset.

The design choices are intentional:
- We keep only the columns needed for the analysis so the notebook stays lighter on memory.
- We use **rolling correlation** as a screening step because testing every possible pair each month becomes expensive as the universe grows.
- We then apply an **Engle-Granger cointegration test** to the shortlisted pairs because correlation alone does not guarantee a mean-reverting spread.
- We trade on the **z-score of the spread** and only refresh the pair list every few months, while still checking trade signals monthly.

Because the JKP file is monthly, this is a **slow-frequency statistical arbitrage** notebook rather than a high-frequency pairs strategy. The signals are coarser, so the thresholds are intentionally wider than you might use with daily data.

In [ ]:
# ── Environment setup (run once) ─────────────────────────
import sys, os, subprocess

if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    if not os.path.exists("quant-trading-experiments"):
        subprocess.run(
            ["git", "clone", "https://github.com/gunsslashroses/quant-trading-experiments.git"],
            check=True,
        )
    %cd quant-trading-experiments
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
    JKP_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/CPSC 381-581: Machine Learning/Final project/jkp_data.csv"
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(repo_root)
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    JKP_CSV_PATH = "/Users/abhin/Library/CloudStorage/GoogleDrive-abhinavkeshri1999@gmail.com/My Drive/Colab Notebooks/CPSC 381-581: Machine Learning/Final project/jkp_data.csv"

print("Using JKP_CSV_PATH:", JKP_CSV_PATH)

## Strategy Logic

The notebook follows a conservative monthly workflow:

1. **Universe construction**
   We keep only stocks with enough monthly history to estimate both correlation and cointegration reliably.

2. **Correlation screen**
   Each rebalance date, we compute trailing return correlations and keep only the strongest candidates. This step is mainly about speed and noise control.

3. **Cointegration filter**
   For those candidates, we run the Engle-Granger test on reconstructed log-price series. A pair can be correlated without having a stable spread, so this is the key statistical filter.

4. **Spread signal**
   For each selected pair, we estimate a hedge ratio on a trailing window, compute the spread, and convert it to a z-score.

5. **Monthly trading rule**
   At the end of month `t`, we use the z-score observed at `t` to decide the position for month `t+1`. This avoids look-ahead bias.

6. **Portfolio construction**
   Active pairs are equally weighted. Within each pair, the long and short sides are scaled by the hedge ratio and then normalized by gross exposure.

A practical note: JKP does not ship raw daily prices here, so we reconstruct a **price index** from monthly returns. That is enough for a monthly cointegration and spread framework, but it is still an approximation relative to using raw security prices.

In [ ]:
import itertools
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
from statsmodels.tsa.stattools import coint
from tqdm.auto import tqdm

from quant_trading.data import load_jkp_csv

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)


def compute_annualized_metrics(returns: pd.Series, freq: int = 12) -> pd.Series:
    """Simple performance summary for a monthly return series."""
    rets = pd.Series(returns).dropna().astype(float)
    if rets.empty:
        return pd.Series(
            {
                "Ann Return": np.nan,
                "Ann Vol": np.nan,
                "Sharpe": np.nan,
                "Max Drawdown": np.nan,
                "Hit Rate": np.nan,
                "Months": 0,
            }
        )

    wealth = (1.0 + rets).cumprod()
    peak = wealth.cummax()
    drawdown = wealth / peak - 1.0

    ann_return = (1.0 + rets.mean()) ** freq - 1.0
    ann_vol = rets.std(ddof=1) * np.sqrt(freq)
    sharpe = np.nan if ann_vol == 0 or np.isnan(ann_vol) else ann_return / ann_vol

    return pd.Series(
        {
            "Ann Return": ann_return,
            "Ann Vol": ann_vol,
            "Sharpe": sharpe,
            "Max Drawdown": drawdown.min(),
            "Hit Rate": (rets > 0).mean(),
            "Months": int(rets.shape[0]),
        }
    )


def estimate_half_life(spread: pd.Series) -> float:
    """Estimate mean-reversion half-life from a simple AR(1)-style regression."""
    s = pd.Series(spread).dropna()
    if len(s) < 12:
        return np.nan

    lagged = s.shift(1).dropna()
    delta = s.diff().dropna()
    aligned = pd.concat([lagged, delta], axis=1).dropna()
    if aligned.empty:
        return np.nan

    y = aligned.iloc[:, 1]
    x = add_constant(aligned.iloc[:, 0])
    beta = OLS(y, x).fit().params.iloc[1]
    if pd.isna(beta) or beta >= 0:
        return np.nan

    half_life = -np.log(2) / beta
    return float(half_life) if np.isfinite(half_life) else np.nan


def fit_spread_model(y_log: pd.Series, x_log: pd.Series) -> dict:
    """Estimate alpha, beta, spread, and half-life for a candidate pair."""
    aligned = pd.concat([y_log.rename("y"), x_log.rename("x")], axis=1).dropna()
    if len(aligned) < 12:
        return {"alpha": np.nan, "beta": np.nan, "spread": pd.Series(dtype=float), "half_life": np.nan}

    model = OLS(aligned["y"], add_constant(aligned["x"])).fit()
    alpha = float(model.params["const"])
    beta = float(model.params["x"])
    spread = aligned["y"] - (alpha + beta * aligned["x"])
    return {
        "alpha": alpha,
        "beta": beta,
        "spread": spread,
        "half_life": estimate_half_life(spread),
    }


def orient_pair_for_cointegration(price_log: pd.DataFrame, left_id: int, right_id: int) -> dict | None:
    """Run Engle-Granger in both directions and keep the stronger orientation.

    The Engle-Granger procedure is asymmetric because the regression picks one stock
    as the dependent variable. Testing both directions makes the candidate ranking
    more robust and gives us a clearer trading spread.
    """
    p_left = price_log[left_id].dropna()
    p_right = price_log[right_id].dropna()
    aligned = pd.concat([p_left.rename("left"), p_right.rename("right")], axis=1).dropna()
    if len(aligned) < 24:
        return None

    results = []
    for y_col, x_col in [("left", "right"), ("right", "left")]:
        try:
            test_stat, pvalue, _ = coint(aligned[y_col], aligned[x_col])
        except Exception:
            continue

        fit = fit_spread_model(aligned[y_col], aligned[x_col])
        results.append(
            {
                "y_id": left_id if y_col == "left" else right_id,
                "x_id": right_id if x_col == "right" else left_id,
                "test_stat": float(test_stat),
                "pvalue": float(pvalue),
                "alpha": fit["alpha"],
                "beta": fit["beta"],
                "half_life": fit["half_life"],
            }
        )

    if not results:
        return None
    return min(results, key=lambda row: row["pvalue"])


def select_candidate_pairs(
    returns_window: pd.DataFrame,
    price_log_window: pd.DataFrame,
    latest_meta: pd.DataFrame,
    top_k_correlated: int,
    coint_pvalue_max: float,
    max_half_life: float,
    same_industry_only: bool,
    max_active_pairs: int,
) -> pd.DataFrame:
    """Screen pairs by correlation first, then cointegration.

    Correlation is only a first-pass filter here. The actual trading candidates are
    the pairs whose spread survives the cointegration screen and has a plausible
    mean-reversion half-life.
    """
    eligible_ids = returns_window.columns[returns_window.notna().all()].tolist()
    if len(eligible_ids) < 2:
        return pd.DataFrame()

    corr_matrix = returns_window[eligible_ids].corr()
    id_to_group = latest_meta.set_index("id")["ff49"].to_dict() if "ff49" in latest_meta.columns else {}

    corr_rows = []
    for left_id, right_id in itertools.combinations(eligible_ids, 2):
        if same_industry_only and id_to_group.get(left_id) != id_to_group.get(right_id):
            continue
        corr_val = corr_matrix.loc[left_id, right_id]
        if pd.isna(corr_val):
            continue
        corr_rows.append({"left_id": left_id, "right_id": right_id, "corr": float(corr_val)})

    if not corr_rows:
        return pd.DataFrame()

    corr_df = pd.DataFrame(corr_rows).sort_values("corr", ascending=False).head(top_k_correlated)

    selected = []
    for row in corr_df.itertuples(index=False):
        oriented = orient_pair_for_cointegration(price_log_window, row.left_id, row.right_id)
        if oriented is None:
            continue
        if oriented["pvalue"] > coint_pvalue_max:
            continue
        if pd.isna(oriented["half_life"]) or oriented["half_life"] <= 0 or oriented["half_life"] > max_half_life:
            continue
        selected.append(
            {
                "y_id": oriented["y_id"],
                "x_id": oriented["x_id"],
                "corr": row.corr,
                "pvalue": oriented["pvalue"],
                "test_stat": oriented["test_stat"],
                "alpha": oriented["alpha"],
                "beta": oriented["beta"],
                "half_life": oriented["half_life"],
            }
        )

    if not selected:
        return pd.DataFrame()

    selected_df = pd.DataFrame(selected)
    selected_df = selected_df.sort_values(["pvalue", "corr"], ascending=[True, False]).head(max_active_pairs)
    return selected_df.reset_index(drop=True)


def compute_pair_signal(price_log_window: pd.DataFrame, y_id: int, x_id: int, zscore_lookback: int) -> dict | None:
    """Estimate the current spread z-score using only data available at the decision date."""
    aligned = pd.concat(
        [price_log_window[y_id].rename("y"), price_log_window[x_id].rename("x")],
        axis=1,
    ).dropna()
    if len(aligned) < max(24, zscore_lookback):
        return None

    fit = fit_spread_model(aligned["y"], aligned["x"])
    spread = fit["spread"].dropna()
    hist = spread.tail(zscore_lookback)
    if len(hist) < zscore_lookback:
        return None

    mean = hist.mean()
    std = hist.std(ddof=1)
    if std == 0 or pd.isna(std):
        return None

    current_spread = float(spread.iloc[-1])
    zscore = (current_spread - mean) / std
    return {
        "alpha": fit["alpha"],
        "beta": fit["beta"],
        "spread": current_spread,
        "zscore": float(zscore),
        "half_life": fit["half_life"],
    }


def pair_return_from_next_month(y_ret: float, x_ret: float, beta: float, signal: int) -> float:
    """Convert next-month stock returns into a gross-normalized pair return.

    signal = +1 means long the spread: long y, short beta units of x.
    signal = -1 means short the spread.
    """
    gross = 1.0 + abs(beta)
    raw_spread_return = y_ret - beta * x_ret
    return signal * (raw_spread_return / gross)


## Configuration

These defaults are chosen for monthly data rather than daily data.

- The lookback windows are fairly long because monthly observations are sparse.
- Pair selection is refreshed every 6 months to reduce turnover in the pair list.
- Z-score thresholds are wider because monthly spreads move in larger jumps than daily spreads.

In [ ]:
START_DATE = "1990-01-01"
END_DATE = None

ID_COL = "id"
DATE_COL = "month_date"
RETURN_COL = "ret"
SIZE_COL = "market_equity"
INDUSTRY_COL = "ff49"

MIN_HISTORY_MONTHS = 60
CORR_LOOKBACK = 60
ZSCORE_LOOKBACK = 24
PAIR_REFRESH_MONTHS = 6

UNIVERSE_SIZE = 80
TOP_K_CORRELATED = 100
MAX_ACTIVE_PAIRS = 5
SAME_INDUSTRY_ONLY = True

COINT_PVALUE_MAX = 0.05
MAX_HALF_LIFE = 24

ENTRY_Z = 1.5
EXIT_Z = 0.5
STOP_Z = 3.0


## Load Only the Columns We Need

This step is intentionally strict about memory use.

The raw JKP file contains many columns that are useful for factor research but unnecessary for a pairs notebook. Here we only keep:
- an identifier,
- the month,
- the realized monthly return,
- market equity for a simple size-based universe filter,
- and optionally `ff49` so we can pair stocks inside the same industry bucket.

We also reconstruct a price index from monthly returns because cointegration operates on price-like series rather than on next-month target returns.

In [ ]:
df_raw = load_jkp_csv(
    JKP_CSV_PATH,
    start_date=START_DATE,
    end_date=END_DATE or "2026-12-31",
)
df_raw = df_raw.loc[:, ~df_raw.columns.duplicated(keep="first")]

required_cols = [ID_COL, DATE_COL, RETURN_COL, SIZE_COL]
optional_cols = [INDUSTRY_COL]
keep_cols = [c for c in required_cols + optional_cols if c in df_raw.columns]

missing_required = [c for c in required_cols if c not in df_raw.columns]
if missing_required:
    raise ValueError(f"Missing required columns in JKP data: {missing_required}")

df = df_raw[keep_cols].copy()
del df_raw

df[DATE_COL] = pd.to_datetime(df[DATE_COL]).values.astype("datetime64[M]")
df = df.dropna(subset=[ID_COL, DATE_COL, RETURN_COL, SIZE_COL]).copy()
df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype("Int64")
df = df.dropna(subset=[ID_COL]).copy()
df[ID_COL] = df[ID_COL].astype(int)
df[RETURN_COL] = pd.to_numeric(df[RETURN_COL], errors="coerce").astype("float32")
df[SIZE_COL] = pd.to_numeric(df[SIZE_COL], errors="coerce").astype("float32")
if INDUSTRY_COL in df.columns:
    df[INDUSTRY_COL] = pd.to_numeric(df[INDUSTRY_COL], errors="coerce").fillna(-1).astype("int16")

# Returns below -100% break log-price reconstruction, so we drop those rows.
df = df[df[RETURN_COL] > -1.0].copy()

# Keep only stocks with enough total monthly observations to support rolling estimation.
history_counts = df.groupby(ID_COL)[DATE_COL].transform("count")
df = df[history_counts >= MIN_HISTORY_MONTHS].copy()

# Reconstruct a price index from realized monthly returns.
df = df.sort_values([ID_COL, DATE_COL]).copy()
df["gross_ret"] = 1.0 + df[RETURN_COL]
df["price_index"] = df.groupby(ID_COL)["gross_ret"].cumprod()
df["log_price"] = np.log(df["price_index"])

print(f"Rows kept: {len(df):,}")
print(f"Unique stocks: {df[ID_COL].nunique():,}")
print(f"Date range: {df[DATE_COL].min().date()} to {df[DATE_COL].max().date()}")
display(df.head())

## Build Compact Panels

The backtest becomes much easier once the data is reshaped into wide monthly panels.

- `returns_panel` is used for rolling correlation screens and for realized next-month PnL.
- `price_log_panel` is used for the cointegration test and spread construction.
- `latest_month_meta` stores only the most recent stock attributes we need at each decision date, mainly size and industry.

To keep runtime manageable, we also cap the universe each month at the largest names by market equity.

In [ ]:
returns_panel = df.pivot(index=DATE_COL, columns=ID_COL, values=RETURN_COL).sort_index()
price_log_panel = df.pivot(index=DATE_COL, columns=ID_COL, values="log_price").sort_index()

meta_cols = [ID_COL, DATE_COL, SIZE_COL] + ([INDUSTRY_COL] if INDUSTRY_COL in df.columns else [])
monthly_meta = df[meta_cols].copy()

latest_month_meta = {
    month: frame.sort_values(SIZE_COL, ascending=False).drop_duplicates(ID_COL)
    for month, frame in monthly_meta.groupby(DATE_COL)
}

all_months = returns_panel.index.tolist()
print("Return panel shape:", returns_panel.shape)
print("Price panel shape:", price_log_panel.shape)

## Rolling Pair Selection and Trading

This is the heart of the notebook.

At each decision month:
- If it is a refresh month, we rebuild the pair list using only trailing data.
- For the current active pairs, we compute a fresh spread z-score.
- We translate that z-score into a stateful trading decision for the next month.

The position logic is intentionally simple:
- Enter **long spread** when the spread is sufficiently cheap (`z < -ENTRY_Z`).
- Enter **short spread** when the spread is sufficiently rich (`z > ENTRY_Z`).
- Exit when the spread has mostly normalized (`|z| < EXIT_Z`).
- Force an exit if the spread blows out too far (`|z| > STOP_Z`).

With monthly data, fewer moving parts is usually better. The goal here is a transparent, believable baseline rather than an over-tuned strategy.

In [ ]:
backtest_rows = []
pair_selection_rows = []
pair_state = {}
active_pairs = pd.DataFrame()

start_idx = max(CORR_LOOKBACK, ZSCORE_LOOKBACK)
decision_months = all_months[start_idx:-1]

for step_idx, month in enumerate(tqdm(decision_months, desc="Backtesting monthly pairs")):
    next_month = all_months[all_months.index(month) + 1]
    lookback_months = all_months[max(0, all_months.index(month) - CORR_LOOKBACK + 1): all_months.index(month) + 1]
    returns_window_full = returns_panel.loc[lookback_months]
    price_log_window_full = price_log_panel.loc[lookback_months]

    current_meta = latest_month_meta.get(month)
    if current_meta is None or current_meta.empty:
        continue

    eligible_ids = current_meta.nlargest(UNIVERSE_SIZE, SIZE_COL)[ID_COL].tolist()
    returns_window = returns_window_full.reindex(columns=eligible_ids)
    price_log_window = price_log_window_full.reindex(columns=eligible_ids)

    refresh_now = (step_idx % PAIR_REFRESH_MONTHS == 0) or active_pairs.empty
    if refresh_now:
        active_pairs = select_candidate_pairs(
            returns_window=returns_window,
            price_log_window=price_log_window,
            latest_meta=current_meta,
            top_k_correlated=TOP_K_CORRELATED,
            coint_pvalue_max=COINT_PVALUE_MAX,
            max_half_life=MAX_HALF_LIFE,
            same_industry_only=SAME_INDUSTRY_ONLY,
            max_active_pairs=MAX_ACTIVE_PAIRS,
        )
        pair_state = {f"{row.y_id}_{row.x_id}": 0 for row in active_pairs.itertuples(index=False)}
        if not active_pairs.empty:
            refresh_snapshot = active_pairs.copy()
            refresh_snapshot.insert(0, "selection_month", month)
            pair_selection_rows.append(refresh_snapshot)

    month_pair_returns = []
    month_pair_details = []

    for row in active_pairs.itertuples(index=False):
        pair_key = f"{row.y_id}_{row.x_id}"
        signal_info = compute_pair_signal(price_log_window, row.y_id, row.x_id, ZSCORE_LOOKBACK)
        if signal_info is None:
            continue

        current_signal = pair_state.get(pair_key, 0)
        zscore = signal_info["zscore"]

        if abs(zscore) >= STOP_Z:
            current_signal = 0
        elif current_signal == 0:
            if zscore <= -ENTRY_Z:
                current_signal = 1
            elif zscore >= ENTRY_Z:
                current_signal = -1
        elif abs(zscore) <= EXIT_Z:
            current_signal = 0

        pair_state[pair_key] = current_signal
        next_y_ret = returns_panel.at[next_month, row.y_id] if row.y_id in returns_panel.columns else np.nan
        next_x_ret = returns_panel.at[next_month, row.x_id] if row.x_id in returns_panel.columns else np.nan
        if current_signal == 0 or pd.isna(next_y_ret) or pd.isna(next_x_ret):
            pair_ret = 0.0
        else:
            pair_ret = pair_return_from_next_month(
                y_ret=float(next_y_ret),
                x_ret=float(next_x_ret),
                beta=float(signal_info["beta"]),
                signal=int(current_signal),
            )

        month_pair_returns.append(pair_ret)
        month_pair_details.append(
            {
                "decision_month": month,
                "return_month": next_month,
                "y_id": row.y_id,
                "x_id": row.x_id,
                "corr": row.corr,
                "coint_pvalue": row.pvalue,
                "beta": signal_info["beta"],
                "zscore": zscore,
                "signal": current_signal,
                "pair_return": pair_ret,
            }
        )

    portfolio_ret = float(np.mean(month_pair_returns)) if month_pair_returns else 0.0
    backtest_rows.append(
        {
            "decision_month": month,
            "return_month": next_month,
            "portfolio_return": portfolio_ret,
            "active_pairs": int(sum(1 for detail in month_pair_details if detail["signal"] != 0)),
            "candidate_pairs": int(len(active_pairs)),
        }
    )
    backtest_rows.extend(month_pair_details)

backtest_df = pd.DataFrame(backtest_rows)
portfolio_results = backtest_df.loc[backtest_df["portfolio_return"].notna(), ["return_month", "portfolio_return", "active_pairs", "candidate_pairs"]].drop_duplicates().sort_values("return_month")
pair_results = backtest_df.loc[backtest_df["pair_return"].notna()].copy() if "pair_return" in backtest_df.columns else pd.DataFrame()
pair_selection_df = pd.concat(pair_selection_rows, ignore_index=True) if pair_selection_rows else pd.DataFrame()

display(portfolio_results.head())
display(pair_selection_df.head())

## Results

The summary below is for the **portfolio of active pairs**, not for individual pair trades.

A few things to watch when you interpret the output:
- If the number of active pairs is often zero, the filters may be too strict for your sample.
- If the Sharpe looks strong but there are very few trades, the result may not be stable.
- If performance deteriorates sharply when you widen the universe or relax same-industry matching, the strategy may be relying heavily on a narrow subset of names.

In [ ]:
metrics = compute_annualized_metrics(portfolio_results["portfolio_return"])
display(metrics.to_frame(name="Monthly Pairs Portfolio"))

if not portfolio_results.empty:
    plot_df = portfolio_results.copy()
    plot_df["cum_return"] = (1.0 + plot_df["portfolio_return"]).cumprod()

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    axes[0].plot(plot_df["return_month"], plot_df["cum_return"], linewidth=2)
    axes[0].set_title("Cumulative Return of Monthly Pairs Portfolio")
    axes[0].set_ylabel("Growth of $1")

    axes[1].plot(plot_df["return_month"], plot_df["active_pairs"], linewidth=1.5)
    axes[1].set_title("Number of Active Pairs")
    axes[1].set_ylabel("Pairs")
    axes[1].set_xlabel("Month")

    plt.tight_layout()
    plt.show()

## Inspect the Selected Pairs

This last section is useful for sanity-checking whether the strategy is selecting intuitively believable pairs.

A healthy workflow is to inspect:
- which pairs are selected most often,
- whether their cointegration p-values remain consistently low,
- and whether the spread z-score behavior looks genuinely mean-reverting rather than episodic.

If you want, the next iteration can add pair-level plots of the reconstructed price series and spread z-score for the most frequently traded pairs.

In [ ]:
if not pair_selection_df.empty:
    pair_counts = (
        pair_selection_df.assign(pair=lambda x: x["y_id"].astype(str) + " vs " + x["x_id"].astype(str))
        .groupby("pair")
        .agg(
            selections=("pair", "size"),
            avg_corr=("corr", "mean"),
            avg_coint_pvalue=("pvalue", "mean"),
            avg_half_life=("half_life", "mean"),
        )
        .sort_values(["selections", "avg_coint_pvalue"], ascending=[False, True])
    )
    display(pair_counts.head(15))
else:
    print("No pairs passed the current correlation and cointegration filters.")